# Notebook 05d — Recommender Rebuild: Cuisine Filter + Popularity Penalty

**Revision note:** the first version of this notebook tried folding cuisine into `W`
as 5 more PCA archetypes, grounded by averaging each genre's songs into a centroid
and doing nearest-neighbor search in Spotify-PC space. Testing that directly showed
it doesn't work: even a *single pure genre's* centroid (e.g. `country` alone, 570
songs) returns mostly unrelated genres as its nearest neighbors (`cantopop`, `trance`,
`alt-rock`...). Genre isn't an audio-feature concept in this dataset, so
averaging-then-nearest-neighbor can't recover it.

**The fix:** genre becomes a **hard filter on the candidate pool**, driven directly by
each restaurant's raw `Cuisine.*` flags (not PCA scores). `W` goes back to just the 8
original vibe archetypes, which still do real work: *ranking* within the
genre-filtered pool.

**This version adds:** a popularity penalty (the other reason we switched datasets --
this catalog has real `popularity`, the old one didn't) and bundles everything the API
needs into one file, `recommender_artifact_v2.pkl`.


In [1]:
import os
import re

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

PROCESSED = r"C:\Users\ryanm\Documents\coding_projects\restaurant_recommendation\data\processed"

songs = pd.read_csv(os.path.join(PROCESSED, "song_pca_new.csv"))
restaurants = pd.read_csv(os.path.join(PROCESSED, "restaurant_pca_new.csv"))
yelp_raw = pd.read_csv(os.path.join(PROCESSED, "yelp_ml_ready_new.csv"))

SPOTIFY_PC_COLS = [c for c in songs.columns if c.startswith("pc")]
song_vecs = songs[SPOTIFY_PC_COLS].values

print(f"Songs:       {songs.shape}")
print(f"Restaurants: {restaurants.shape}")


Songs:       (81198, 11)
Restaurants: (34808, 27)


## Step 1 — `W`: the 8 original vibe archetypes


In [2]:
YELP_PC_LABELS = {
    "pc1":  "sit-down dinner",
    "pc3":  "brunch / breakfast",
    "pc8":  "hipster / trendy",
    "pc11": "dive bar",
    "pc17": "late-night",
    "pc18": "good for groups",
    "pc20": "loud / chaotic, lower-rated",
    "pc22": "fine dining",
}
ARCHETYPE_PCS = list(YELP_PC_LABELS.keys())

SPOTIFY_PC_LABELS = {
    "pc1": "energetic + loud + danceable",
    "pc2": "happy + danceable (acoustic-leaning)",
    "pc3": "live performance + speechy",
    "pc4": "fast tempo",
    "pc5": "spoken word / pure instrumental",
    "pc6": "live instrumental / jazz-like",
}

restaurant_vecs = restaurants[ARCHETYPE_PCS].values
print(f"Restaurant vectors (covered PCs only): {restaurant_vecs.shape}")


Restaurant vectors (covered PCs only): (34808, 8)


In [3]:
ANCHOR_ARTISTS = {
    "pc1":  ["Bruno Mars", "John Legend", "Michael Buble", "Jason Mraz", "Andra Day",
             "Alicia Keys", "Adele", "Ben Rector", "Colbie Caillat", "Gavin DeGraw"],
    "pc3":  ["Leon Bridges", "Norah Jones", "Jack Johnson", "John Mayer", "The Lumineers",
             "Maggie Rogers", "Vance Joy", "Kacey Musgraves", "Fleet Foxes", "Hozier"],
    "pc22": ["Miles Davis", "John Coltrane", "Frank Sinatra", "Ella Fitzgerald", "Bill Evans",
             "Sade", "Diana Krall", "Nat King Cole", "Stan Getz", "Nina Simone"],
    "pc20": ["AC/DC", "Guns N' Roses", "Red Hot Chili Peppers", "Kings of Leon",
             "The White Stripes", "Foo Fighters", "Weezer", "blink-182", "Fall Out Boy", "Panic! At The Disco"],
    "pc8":  ["Tame Impala", "Khruangbin", "Thundercat", "Vampire Weekend", "Mac DeMarco",
             "Arctic Monkeys", "Phoebe Bridgers", "LCD Soundsystem", "The Strokes", "Blood Orange"],
    "pc11": ["Tom Waits", "The Black Keys", "Alabama Shakes", "Gary Clark Jr.", "ZZ Top",
             "Jack White", "The Rolling Stones", "Bob Seger", "Creedence Clearwater Revival", "Whiskey Myers"],
    "pc17": ["FKA twigs", "Billie Eilish", "Frank Ocean", "James Blake", "Rhye",
             "Bon Iver", "Daniel Caesar", "SZA", "Jorja Smith", "Lianne La Havas"],
    "pc18": ["Daft Punk", "Disclosure", "Calvin Harris", "The Weeknd", "Dua Lipa",
             "Fred again..", "RUFUS DU SOL", "KAYTRANADA", "Jamie xx", "Peggy Gou"],
}

MIN_ANCHOR_SONGS = 5
Y_train_by_pc = {}

for pc, artists in ANCHOR_ARTISTS.items():
    mask = pd.Series(False, index=songs.index)
    for artist in artists:
        mask |= songs["artists"].str.contains(re.escape(artist), case=False, na=False)
    n = int(mask.sum())
    print(f"{pc} ({YELP_PC_LABELS[pc]}): {n} matched songs")
    assert n >= MIN_ANCHOR_SONGS, f"{pc} fell below the anchor-song floor"
    Y_train_by_pc[pc] = song_vecs[mask.values].mean(axis=0)

X_train = np.eye(len(ARCHETYPE_PCS))
Y_train = np.array([Y_train_by_pc[pc] for pc in ARCHETYPE_PCS])
W = LinearRegression(fit_intercept=False).fit(X_train, Y_train).coef_

W_df = pd.DataFrame(W, index=SPOTIFY_PC_COLS, columns=ARCHETYPE_PCS)
print()
print("Weight matrix W:")
print(W_df.round(3).to_string())


pc1 (sit-down dinner): 130 matched songs


pc3 (brunch / breakfast): 119 matched songs


pc22 (fine dining): 115 matched songs


pc20 (loud / chaotic, lower-rated): 199 matched songs


pc8 (hipster / trendy): 169 matched songs


pc11 (dive bar): 78 matched songs


pc17 (late-night): 131 matched songs


pc18 (good for groups): 115 matched songs

Weight matrix W:
       pc1    pc3    pc8   pc11   pc17   pc18   pc20   pc22
pc1 -0.137 -0.671  0.549  0.602 -0.424  0.975  1.059 -1.508
pc2  0.347  0.476 -0.314  0.226  0.395  0.287 -0.512  0.867
pc3 -0.240 -0.211 -0.393 -0.337  0.220 -0.364 -0.270  0.209
pc4 -0.447 -0.586 -0.142 -0.269 -0.001  0.204 -0.246 -0.652
pc5 -0.312 -0.266 -0.286 -0.275  0.033 -0.173 -0.321 -0.091
pc6 -0.445 -0.324  0.065  0.000 -0.451 -0.144 -0.139 -0.338


## Step 2 — Cuisine-to-genre filter map


In [4]:
CUISINE_GENRE_FILTERS = {
    "Cuisine.Southern":      ["country", "honky-tonk", "gospel", "bluegrass"],
    "Cuisine.SteakhouseBBQ": ["country", "honky-tonk", "gospel", "bluegrass", "blues"],
    "Cuisine.TexMex":        ["latin", "latino", "salsa", "reggaeton", "spanish"],
    "Cuisine.Mexican":       ["latin", "latino", "salsa", "reggaeton", "spanish"],
    "Cuisine.LatinAmerican": ["latin", "latino", "salsa", "reggaeton", "spanish",
                              "samba", "forro", "sertanejo", "mpb", "pagode", "tango"],
    "Cuisine.Chinese":       ["cantopop", "mandopop"],
    "Cuisine.Japanese":      ["j-pop", "j-rock", "j-idol", "j-dance", "anime"],
    "Cuisine.SushiBars":     ["j-pop", "j-rock", "j-idol", "j-dance", "anime"],
    "Cuisine.Indian":        ["indian"],
}

for col, genres in CUISINE_GENRE_FILTERS.items():
    n = songs["track_genre"].isin(genres).sum()
    print(f"{col:22s} -> {genres}  ({n} matching songs in catalog)")


Cuisine.Southern       -> ['country', 'honky-tonk', 'gospel', 'bluegrass']  (3039 matching songs in catalog)
Cuisine.SteakhouseBBQ  -> ['country', 'honky-tonk', 'gospel', 'bluegrass', 'blues']  (3725 matching songs in catalog)
Cuisine.TexMex         -> ['latin', 'latino', 'salsa', 'reggaeton', 'spanish']  (2633 matching songs in catalog)
Cuisine.Mexican        -> ['latin', 'latino', 'salsa', 'reggaeton', 'spanish']  (2633 matching songs in catalog)
Cuisine.LatinAmerican  -> ['latin', 'latino', 'salsa', 'reggaeton', 'spanish', 'samba', 'forro', 'sertanejo', 'mpb', 'pagode', 'tango']  (7522 matching songs in catalog)
Cuisine.Chinese        -> ['cantopop', 'mandopop']  (1798 matching songs in catalog)


Cuisine.Japanese       -> ['j-pop', 'j-rock', 'j-idol', 'j-dance', 'anime']  (3636 matching songs in catalog)
Cuisine.SushiBars      -> ['j-pop', 'j-rock', 'j-idol', 'j-dance', 'anime']  (3636 matching songs in catalog)
Cuisine.Indian         -> ['indian']  (722 matching songs in catalog)


## Step 3 — Popularity penalty (obscure-song avoidance)

Same shape as the old hub-penalty formula (`sqrt` scaling, only kicks in past a
threshold) but inverted: instead of penalizing songs that show up *too often*
(no longer computable the same way against this smaller catalog), this penalizes
songs that are **too obscure** -- now that `popularity` is a real column instead of
a proxy.

`POPULARITY_FLOOR = 10`: about 20% of this catalog sits at or below that (`popularity`
ranges 0-100, mean ~33). Below the floor, the penalty scales up smoothly as
popularity drops toward 0; at/above the floor, no penalty at all. Continuous at the
boundary (both formulas equal 1.0 exactly at `popularity == 10`).


In [5]:
POPULARITY_FLOOR = 10

def popularity_penalty(popularity, floor=POPULARITY_FLOOR):
    pop = np.asarray(popularity, dtype=float)
    penalty = np.ones_like(pop)
    obscure = pop < floor
    penalty[obscure] = np.sqrt(floor / (pop[obscure] + 1))
    return penalty

pop_penalty = popularity_penalty(songs["popularity"].values)

print(f"Songs penalized (popularity < {POPULARITY_FLOOR}): {(pop_penalty > 1).sum()} / {len(songs)}")
print(f"Penalty range: {pop_penalty.min():.2f} - {pop_penalty.max():.2f}")
print()
for p in [0, 5, 9, 10, 20, 50, 100]:
    print(f"  popularity={p:3d}  ->  penalty={popularity_penalty(np.array([p]))[0]:.3f}")


Songs penalized (popularity < 10): 8520 / 81198
Penalty range: 1.00 - 3.16

  popularity=  0  ->  penalty=3.162
  popularity=  5  ->  penalty=1.291
  popularity=  9  ->  penalty=1.000
  popularity= 10  ->  penalty=1.000
  popularity= 20  ->  penalty=1.000
  popularity= 50  ->  penalty=1.000
  popularity=100  ->  penalty=1.000


## Step 4 — Recommend function: genre filters the pool, `W` + popularity rank within it


In [6]:
song_scaler = StandardScaler().fit(song_vecs)
song_vecs_scaled = song_scaler.transform(song_vecs)

def matched_genres_for(cuisine_row):
    genres = set()
    for col, glist in CUISINE_GENRE_FILTERS.items():
        if cuisine_row.get(col, 0) == 1:
            genres.update(glist)
    return genres

def recommend_songs(yelp_vibe_vector, cuisine_row=None, k=5, use_popularity_penalty=True):
    normalized = yelp_vibe_vector / (np.linalg.norm(yelp_vibe_vector) + 1e-9)
    target = song_scaler.transform((W @ normalized).reshape(1, -1))

    genres = matched_genres_for(cuisine_row) if cuisine_row is not None else set()
    if genres:
        candidate_idx = np.where(songs["track_genre"].isin(genres).values)[0]
    else:
        candidate_idx = np.arange(len(songs))

    dists = np.linalg.norm(song_vecs_scaled[candidate_idx] - target, axis=1)
    if use_popularity_penalty:
        dists = dists * pop_penalty[candidate_idx]

    top_local = np.argsort(dists)[:k]
    top_idx = candidate_idx[top_local]

    return pd.DataFrame({
        "song": songs.loc[top_idx, "name"].values,
        "artist": songs.loc[top_idx, "artists"].values,
        "genre": songs.loc[top_idx, "track_genre"].values,
        "popularity": songs.loc[top_idx, "popularity"].values,
        "distance": dists[top_local].round(3),
    })


## Step 5 — Before/after: does the popularity penalty actually change anything?


In [7]:
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(restaurants), size=15, replace=False)

changed = 0
for idx in sample_idx:
    y = restaurant_vecs[idx]
    biz_id = restaurants.loc[idx, "business_id"]
    cuisine_row = yelp_raw[yelp_raw["business_id"] == biz_id].iloc[0]

    before = recommend_songs(y, cuisine_row=cuisine_row, k=3, use_popularity_penalty=False)["song"].tolist()
    after = recommend_songs(y, cuisine_row=cuisine_row, k=3, use_popularity_penalty=True)["song"].tolist()
    changed += before != after

print(f"{changed} / {len(sample_idx)} sampled restaurants had their top-3 change once the popularity penalty was applied")


1 / 15 sampled restaurants had their top-3 change once the popularity penalty was applied


## Step 6 — Boss-level check: 8 original vibes


In [8]:
results = []
for pc in ARCHETYPE_PCS:
    idx = restaurants[pc].idxmax()
    recs = recommend_songs(restaurant_vecs[idx], k=3)
    recs.insert(0, "restaurant", restaurants.loc[idx, "name"])
    recs.insert(1, "archetype", YELP_PC_LABELS[pc])
    results.append(recs)

pd.concat(results, ignore_index=True)


,restaurant,archetype,song,artist,genre,popularity,distance
0,The BAO,sit-down dinner,行走的鱼,LaLa Hsu,mandopop,43,0.213
1,The BAO,sit-down dinner,Magic,Cheyenne Jackson;Kerry Butler;Curtis Holbrook;...,show-tunes,22,0.247
2,The BAO,sit-down dinner,Preciso Confiar,Stella Laura;Valesca Mayssa,brazil,46,0.258
3,Living Room Coffee & Kitchen,brunch / breakfast,Táxi Lunar (feat. Geraldo Azevedo),Zé Ramalho;Geraldo Azevedo,mpb,40,0.157
4,Living Room Coffee & Kitchen,brunch / breakfast,Film Credits,Club Kuru,garage,49,0.185
5,Living Room Coffee & Kitchen,brunch / breakfast,Goodbye Yellow Brick Road - Remastered 2014,Elton John,piano,74,0.205
6,Good Fortune,hipster / trendy,Do I Have To Say The Words?,Bryan Adams,singer-songwriter,59,0.220
7,Good Fortune,hipster / trendy,Here Comes The Sun - Remastered 2009,The Beatles,british,82,0.233
8,Good Fortune,hipster / trendy,Shelf In The Room,Days Of The New,grunge,28,0.281
9,Leonardo's Mexican Food,dive bar,Symphony (feat. Zara Larsson),Clean Bandit;Zara Larsson,dance,76,0.111


## Step 7 — Direct check: the Texas-BBQ case


In [9]:
bbq_southern = yelp_raw[(yelp_raw["Cuisine.SteakhouseBBQ"] == 1) & (yelp_raw["Cuisine.Southern"] == 1)]
print(f"Restaurants tagged both BBQ/Steakhouse AND Southern: {len(bbq_southern)}")

sample_id = bbq_southern.iloc[0]["business_id"]
rest_row = restaurants[restaurants["business_id"] == sample_id].iloc[0]
cuisine_row = yelp_raw[yelp_raw["business_id"] == sample_id].iloc[0]

print(f"\nTesting: {rest_row['name']}")
y = rest_row[ARCHETYPE_PCS].values.astype(float)
recommend_songs(y, cuisine_row=cuisine_row, k=5)


Restaurants tagged both BBQ/Steakhouse AND Southern: 195

Testing: Smokin' Out


,song,artist,genre,popularity,distance
0,Metas e Versos (feat. Brunno Ramos),DJ Roger Vale;Zack Vox;Gabriel Bulian;Brunno R...,gospel,40,0.513
1,Bloodshot,Sam Tinnesz,blues,59,0.559
2,Meu Lugar,Matheus Balo;Filipe Lancaster,gospel,41,0.561
3,When Everything Went Wrong (from the series Ar...,Fantastic Negrito,blues,55,0.564
4,He Bore It All,Dailey & Vincent,bluegrass,24,0.592


## Step 8 — Save the complete API-ready bundle

Everything `api_v2.py` needs in one file: `W`, labels, cuisine filters, popularity
penalty (precomputed per-song, not recomputed per-request), and metadata tables for
both songs and restaurants (including each restaurant's raw `Cuisine.*` flags, since
the API needs those to run the genre filter without re-loading the raw CSVs).

Saved as `recommender_artifact_v2.pkl` -- a new file, alongside the existing
`recommender_artifact.pkl` -- so the API can switch between them via an env var
rather than replacing the old one outright.


In [10]:
CUISINE_COLS = list(CUISINE_GENRE_FILTERS.keys())

songs_meta = songs[["id", "name", "artists", "track_genre", "popularity"]].copy()

restaurants_meta = restaurants[["business_id", "name"] + ARCHETYPE_PCS].copy()
restaurants_meta = restaurants_meta.merge(
    yelp_raw[["business_id"] + CUISINE_COLS], on="business_id", how="left"
)

artifact_v2 = {
    "built_at": pd.Timestamp.now().strftime("%Y-%m-%dT%H:%M:%S"),
    "W": W,
    "archetype_pcs": ARCHETYPE_PCS,
    "spotify_pc_cols": SPOTIFY_PC_COLS,
    "yelp_pc_labels": YELP_PC_LABELS,
    "spotify_pc_labels": SPOTIFY_PC_LABELS,
    "cuisine_genre_filters": CUISINE_GENRE_FILTERS,
    "cuisine_cols": CUISINE_COLS,
    "song_scaler": song_scaler,
    "song_vecs_scaled": song_vecs_scaled,
    "popularity_penalty": pop_penalty,
    "popularity_floor": POPULARITY_FLOOR,
    "songs_meta": songs_meta,
    "restaurants_meta": restaurants_meta,
}

joblib.dump(artifact_v2, os.path.join(PROCESSED, "recommender_artifact_v2.pkl"))
print(f"Saved recommender_artifact_v2.pkl")
print(f"  restaurants_meta: {restaurants_meta.shape}")
print(f"  songs_meta:       {songs_meta.shape}")


Saved recommender_artifact_v2.pkl
  restaurants_meta: (34808, 19)
  songs_meta:       (81198, 5)
